# 03 — Model Training

Trains, compares, refits and tests candidate-road models. **Requires:** notebook 02 outputs and `src/models/candidate_models.py`.

In [ ]:
import joblib
import numpy as np
import pandas as pd
pd.set_option("display.max_columns", None)


## Train and Compare Candidate-Road Models

This section compares Logistic Regression, Random Forest, Linear SVM, XGBoost, and KNN using trip-held-out validation data. The primary selection metric is observation-level Top-1 accuracy: for each GPS observation, it chooses the highest-scored road candidate and checks it against the reference segment. Route continuity and candidate-level F1 break ties. Student-test data is not used for model selection. Training is deterministically capped at 500 observations per training trip to keep memory bounded; SVM calibration uses a separate subset of training observations, and KNN uses at most 20 observations per trip.

In [ ]:
from pathlib import Path
import sys

working_directory = Path.cwd().resolve()
root_candidates = [working_directory, *working_directory.parents]
root_candidates += [parent / 'PathSnap' for parent in root_candidates]
project_root = next(
    (candidate for candidate in root_candidates
     if (candidate / 'src' / 'models' / 'candidate_models.py').is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError('Could not locate src/models/candidate_models.py from the notebook working directory.')
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.models.candidate_models import train_candidate_models

feature_root = project_root / 'data' / 'processed' / 'features'
cleaned_root = project_root / 'data' / 'processed' / 'cleaned'
model_output_root = project_root / 'results' / 'candidate_models'

model_comparison, best_model_name, best_model_path, model_run_summary = train_candidate_models(
    feature_root=feature_root,
    cleaned_root=cleaned_root,
    output_root=model_output_root,
    max_train_observations_per_trip=500,
    calibration_fraction=0.15,
    knn_observations_per_trip=20,
    random_seed=42,
)
display(model_comparison)
print(f'Best model by validation observation-level Top-1 accuracy: {best_model_name}')
print(f'Selected model saved to: {best_model_path}')
print(f'Run summary: {model_run_summary}')

### Refit the Selected Model on All Training-Partition Trips

After selecting the model using the 64/16 train-validation split, refit only that model on the full training partition (80 trips, including the former validation trips). This uses no student-test features or labels. The deterministic per-trip cap remains in place for memory control; the 500-observation cap applies except for KNN, which remains capped at 20.

In [ ]:
import gc
import json

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from src.models.candidate_models import _read_candidates
from xgboost import XGBClassifier

selection_bundle = joblib.load(best_model_path)
selected_model_name = selection_bundle['model_name']
training_audit = pd.read_csv(feature_root / 'feature_quality_audit.csv', dtype={'trip_id': str})
all_training_trip_ids = sorted(training_audit.loc[training_audit['partition'] == 'train', 'trip_id'].unique())
rng = np.random.default_rng(42)
features = selection_bundle['feature_columns']
train_columns = features + ['observation_id', 'supervision_usable', 'is_target']
fit_x_parts = []
fit_y_parts = []
calibration_x_parts = []
calibration_y_parts = []
final_refit_training_observations = 0
final_refit_calibration_observations = 0
final_refit_trip_ids = []

for trip_id in all_training_trip_ids:
    frame = _read_candidates(feature_root / 'train' / trip_id / 'candidates.csv.gz', train_columns)
    frame = frame.loc[frame['supervision_usable']]
    observation_ids = frame['observation_id'].drop_duplicates().to_numpy()
    if not len(observation_ids):
        continue
    observation_cap = 20 if selected_model_name == 'KNN' else 500
    selected_ids = np.sort(rng.choice(observation_ids, size=min(observation_cap, len(observation_ids)), replace=False))
    frame = frame.loc[frame['observation_id'].isin(selected_ids)]
    calibration_count = 0
    calibration_ids = set()
    if selected_model_name == 'Linear SVM' and len(selected_ids) >= 10:
        calibration_count = max(1, int(round(len(selected_ids) * 0.15)))
        shuffled_ids = rng.permutation(selected_ids)
        calibration_ids = set(shuffled_ids[:calibration_count])
    calibration_mask = frame['observation_id'].isin(calibration_ids).to_numpy()
    fit_mask = ~calibration_mask
    x = frame[features].to_numpy(dtype=np.float32)
    y = frame['is_target'].to_numpy(dtype=np.uint8)
    fit_x_parts.append(x[fit_mask])
    fit_y_parts.append(y[fit_mask])
    calibration_x_parts.append(x[calibration_mask])
    calibration_y_parts.append(y[calibration_mask])
    final_refit_training_observations += len(selected_ids) - calibration_count
    final_refit_calibration_observations += calibration_count
    final_refit_trip_ids.append(trip_id)
    del frame, x, y

if not fit_x_parts:
    raise ValueError('No usable labeled observations found in the full training partition.')
x_fit = np.concatenate(fit_x_parts)
y_fit = np.concatenate(fit_y_parts)
del fit_x_parts, fit_y_parts
final_scaler = StandardScaler(copy=False)
x_fit = final_scaler.fit_transform(x_fit).astype(np.float32, copy=False)
final_calibrator = None

if selected_model_name == 'Logistic Regression':
    final_estimator = LogisticRegression(C=1.0, max_iter=200, solver='lbfgs')
elif selected_model_name == 'Random Forest':
    final_estimator = RandomForestClassifier(n_estimators=50, max_depth=10, min_samples_leaf=10, max_features='sqrt', n_jobs=2, random_state=42)
elif selected_model_name == 'Linear SVM':
    if not calibration_x_parts:
        raise ValueError('Linear SVM final refit requires held-out training observations for calibration.')
    x_calibration = np.concatenate(calibration_x_parts)
    y_calibration = np.concatenate(calibration_y_parts)
    if np.unique(y_calibration).size != 2:
        raise ValueError('SVM calibration data must contain both classes.')
    x_calibration = final_scaler.transform(x_calibration).astype(np.float32, copy=False)
    final_estimator = LinearSVC(C=0.25, dual='auto', max_iter=3000, random_state=42)
elif selected_model_name == 'XGBoost':
    final_estimator = XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.08, subsample=0.8, colsample_bytree=0.8, min_child_weight=5, reg_lambda=2.0, objective='binary:logistic', eval_metric='logloss', tree_method='hist', n_jobs=2, random_state=42)
elif selected_model_name == 'KNN':
    final_estimator = KNeighborsClassifier(n_neighbors=15, weights='distance', algorithm='kd_tree', leaf_size=40, n_jobs=2)
else:
    raise ValueError(f'Unsupported selected model for final refit: {selected_model_name}')

final_estimator.fit(x_fit, y_fit)
if selected_model_name == 'Linear SVM':
    calibration_scores = final_estimator.decision_function(x_calibration).reshape(-1, 1)
    final_calibrator = LogisticRegression(max_iter=200, solver='lbfgs')
    final_calibrator.fit(calibration_scores, y_calibration)

final_bundle = {
    'model_name': selected_model_name,
    'estimator': final_estimator,
    'scaler': final_scaler,
    'calibrator': final_calibrator,
    'feature_columns': features,
    'positive_class': True,
    'classification_threshold': 0.5,
    'final_refit_training_trip_count': len(final_refit_trip_ids),
    'final_refit_training_observations': final_refit_training_observations,
    'final_refit_calibration_observations': final_refit_calibration_observations,
}
joblib.dump(final_bundle, best_model_path)
selection_path = model_output_root / 'model_selection.json'
selection_summary = json.loads(selection_path.read_text(encoding='utf-8'))
selection_summary.update({
    'final_refit_training_partition': 'all training-partition trips after validation-based model selection',
    'final_refit_training_trip_count': len(final_refit_trip_ids),
    'final_refit_training_observations': final_refit_training_observations,
    'final_refit_calibration_observations': final_refit_calibration_observations,
    'final_refit_max_observations_per_trip': 20 if selected_model_name == 'KNN' else 500,
    'student_test_used_for_training': False,
})
selection_path.write_text(json.dumps(selection_summary, indent=2), encoding='utf-8')
print(f'Refit {selected_model_name} on {len(final_refit_trip_ids)} training-partition trips and {final_refit_training_observations:,} sampled observations.')
print(f'Updated final model: {best_model_path}')
del x_fit, y_fit, calibration_x_parts, calibration_y_parts
_ = gc.collect()


### Final Evaluation on the Separate Student-Test Partition


In [ ]:
import time

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, log_loss, precision_score, recall_score
from src.models.candidate_models import _probabilities, _read_candidates, _route_scores

best_bundle = joblib.load(best_model_path)
test_audit = pd.read_csv(feature_root / 'feature_quality_audit.csv', dtype={'trip_id': str})
test_trip_ids = sorted(test_audit.loc[test_audit['partition'] == 'student_test', 'trip_id'].unique())
test_columns = best_bundle['feature_columns'] + [
    'observation_id', 'candidate_arc_id', 'supervision_usable', 'is_target',
    'label_confident', 'label_unambiguous', 'candidate_label_found',
    'route_geometry_suspect', 'route_sequence_suspect', 'route_sequence_consistent',
]
test_frames = []
test_arc_endpoints = {}
test_route_positions = {}
test_candidate_misses = 0
test_reference_count = 0
unscored_test_trips = []

for trip_id in test_trip_ids:
    candidates_path = feature_root / 'student_test' / trip_id / 'candidates.csv.gz'
    frame = _read_candidates(candidates_path, test_columns)
    observation_metadata = frame.drop_duplicates('observation_id')
    reliable_reference = (
        observation_metadata['label_confident']
        & observation_metadata['label_unambiguous']
        & observation_metadata['route_sequence_consistent']
        & ~observation_metadata['route_geometry_suspect']
        & ~observation_metadata['route_sequence_suspect']
    )
    test_reference_count += int(reliable_reference.sum())
    test_candidate_misses += int((reliable_reference & ~observation_metadata['candidate_label_found']).sum())
    usable = frame.loc[frame['supervision_usable']].copy()
    if usable.empty:
        unscored_test_trips.append(trip_id)
        continue
    usable['trip_id'] = trip_id
    test_frames.append(usable[best_bundle['feature_columns'] + ['trip_id', 'observation_id', 'candidate_arc_id', 'is_target']].reset_index(drop=True))
    arc_table = pd.read_csv(cleaned_root / 'student_test' / trip_id / 'arcs.csv', usecols=['arc_id', 'from_node', 'to_node'])
    test_arc_endpoints[trip_id] = {int(row.arc_id): (int(row.from_node), int(row.to_node)) for row in arc_table.itertuples(index=False)}
    route_ids = pd.read_csv(cleaned_root / 'student_test' / trip_id / 'route.csv', usecols=['arc_id'])['arc_id'].to_numpy(dtype=int)
    test_route_positions[trip_id] = {}
    for position, arc_id in enumerate(route_ids):
        test_route_positions[trip_id].setdefault(int(arc_id), position)

if not test_frames:
    raise ValueError('No usable labeled observations were found in the student-test partition.')
test_y_parts = []
test_probability_parts = []
test_top_prediction_parts = []
test_inference_seconds = 0.0
for frame in test_frames:
    start = time.perf_counter()
    probabilities = _probabilities(
        {'name': best_bundle['model_name'], 'estimator': best_bundle['estimator'], 'calibrator': best_bundle['calibrator']},
        frame, best_bundle['scaler'], best_bundle['feature_columns'],
    )
    test_inference_seconds += time.perf_counter() - start
    test_y_parts.append(frame['is_target'].to_numpy(dtype=np.uint8))
    test_probability_parts.append(probabilities.astype(np.float32, copy=False))
    score_frame = pd.DataFrame({'observation_id': frame['observation_id'].to_numpy(), 'score': probabilities})
    top_indices = score_frame.groupby('observation_id', sort=False)['score'].idxmax().to_numpy(dtype=int)
    top = frame.iloc[top_indices][['trip_id', 'observation_id', 'candidate_arc_id', 'is_target']].copy()
    top['is_correct'] = top['is_target'].astype(bool)
    test_top_prediction_parts.append(top)

test_y_true = np.concatenate(test_y_parts)
test_probabilities = np.concatenate(test_probability_parts).astype(float)
test_predictions = test_probabilities >= 0.5
test_top_predictions = pd.concat(test_top_prediction_parts, ignore_index=True)
test_continuity, test_sequence_order, test_sequence_exact = _route_scores(
    test_top_predictions, test_arc_endpoints, test_route_positions
)
best_model_test_metrics = {
    'selected_model': best_bundle['model_name'],
    'scored_test_trips': len(test_frames),
    'unscored_test_trips': ', '.join(unscored_test_trips),
    'accuracy': accuracy_score(test_y_true, test_predictions),
    'precision': precision_score(test_y_true, test_predictions, zero_division=0),
    'recall': recall_score(test_y_true, test_predictions, zero_division=0),
    'f1': f1_score(test_y_true, test_predictions, zero_division=0),
    'confusion_matrix': confusion_matrix(test_y_true, test_predictions, labels=[0, 1]).tolist(),
    'log_loss': log_loss(test_y_true, np.clip(test_probabilities, 1e-7, 1 - 1e-7), labels=[0, 1]),
    'observation_top1_accuracy': float(test_top_predictions['is_correct'].mean()),
    'candidate_miss_rate': test_candidate_misses / test_reference_count if test_reference_count else float('nan'),
    'candidate_misses': test_candidate_misses,
    'reliable_reference_observations': test_reference_count,
    'route_continuity': test_continuity,
    'sequence_order_consistency': test_sequence_order,
    'sequence_exact_trip_rate': test_sequence_exact,
    'evaluated_observations': int(sum(frame['observation_id'].nunique() for frame in test_frames)),
    'evaluated_candidate_rows': int(len(test_y_true)),
    'inference_seconds': test_inference_seconds,
    'model_selection_used_test_data': False,
}
test_metrics_path = model_output_root / 'best_model_test_metrics.csv'
pd.Series(best_model_test_metrics, name='student_test').to_csv(test_metrics_path, header=True)
display(pd.Series(best_model_test_metrics, name='student_test').to_frame())
print(f'Final test metrics saved to: {test_metrics_path}')